# Task 10: Movie Recommendation System

**Content-Based → Hybrid (Content + Collaborative Filtering)**

Dataset: TMDB 5000 Movies

In [1]:
import pandas as pd
import numpy as np
import ast, json
import warnings; warnings.filterwarnings("ignore")
import matplotlib.pyplot as plt
import seaborn as sns

movies  = pd.read_csv("../data/tmdb_5000_movies.csv")
credits = pd.read_csv("../data/tmdb_5000_credits.csv")

print("Movies:", movies.shape)
print("Credits:", credits.shape)
movies.head(2)

Movies: (4803, 20)
Credits: (4803, 4)


,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800
1,300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",http://disney.go.com/disneypictures/pirates/,285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2007-05-19,961000000,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500


## Step 1: Data Preprocessing & Feature Extraction

In [2]:
# Merge on title
credits.rename(columns={"movie_id":"id"}, inplace=True)
df = movies.merge(credits, on="id", how="left")
print("Merged:", df.shape)

def safe_eval(x):
    try: return ast.literal_eval(x)
    except: return []

def extract_names(obj_list, key="name", limit=None):
    items = safe_eval(obj_list) if isinstance(obj_list, str) else obj_list
    names = [i[key] for i in items if key in i]
    return names[:limit] if limit else names

# Extract genres, keywords, cast (top 3), director
df["genres_list"]   = df["genres"].apply(lambda x: extract_names(x))
df["keywords_list"] = df["keywords"].apply(lambda x: extract_names(x))
df["cast_list"]     = df["cast"].apply(lambda x: extract_names(x, limit=3))

def get_director(crew_str):
    crew = safe_eval(crew_str) if isinstance(crew_str, str) else []
    for c in crew:
        if c.get("job") == "Director":
            return [c["name"].replace(" ","")]
    return []

df["director"] = df["crew"].apply(get_director)

# Clean overview
df["overview"] = df["overview"].fillna("")
print("Features extracted")
df[["title_x","genres_list","cast_list","director"]].head(3)

Merged: (4803, 23)
Features extracted


,title_x,genres_list,cast_list,director
0,Avatar,"[Action, Adventure, Fantasy, Science Fiction]","[Sam Worthington, Zoe Saldana, Sigourney Weaver]",[JamesCameron]
1,Pirates of the Caribbean: At World's End,"[Adventure, Fantasy, Action]","[Johnny Depp, Orlando Bloom, Keira Knightley]",[GoreVerbinski]
2,Spectre,"[Action, Adventure, Crime]","[Daniel Craig, Christoph Waltz, Léa Seydoux]",[SamMendes]


## Step 2: Content-Based Filtering (TF-IDF + Cosine Similarity)

In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Build "soup" — combine all metadata into one text blob
def build_soup(row):
    genres   = " ".join(row["genres_list"])
    keywords = " ".join(row["keywords_list"][:5])
    cast     = " ".join([c.replace(" ","") for c in row["cast_list"]])
    director = " ".join(row["director"]) * 3  # weight director higher
    overview = row["overview"]
    return f"{genres} {keywords} {cast} {director} {overview}"

df["soup"] = df.apply(build_soup, axis=1)

# TF-IDF on soup
tfidf = TfidfVectorizer(stop_words="english", max_features=5000)
tfidf_matrix = tfidf.fit_transform(df["soup"])
print(f"TF-IDF matrix: {tfidf_matrix.shape}")

# Cosine similarity matrix
cos_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)
print(f"Similarity matrix: {cos_sim.shape}")

# Index lookup
indices = pd.Series(df.index, index=df["title_x"]).drop_duplicates()
print("Content-Based engine ready")

TF-IDF matrix: (4803, 5000)
Similarity matrix: (4803, 4803)
Content-Based engine ready


In [5]:
def content_recommend(title, top_n=10):
    """Content-Based: recommend movies similar to given title"""
    if title not in indices:
        return f"Movie '{title}' not found in database"
    idx = indices[title]
    sim_scores = list(enumerate(cos_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)[1:top_n+1]
    movie_idxs = [s[0] for s in sim_scores]
    return df[["title_x","genres_list","vote_average"]].iloc[movie_idxs].assign(
        similarity=[round(s[1],3) for s in sim_scores]
    ).rename(columns={"title_x":"title","genres_list":"genres","vote_average":"rating"})

print("Content-Based Recommendations for 'The Dark Knight':")
print(content_recommend("The Dark Knight", top_n=8).to_string(index=False))

Content-Based Recommendations for 'The Dark Knight':
                                  title                           genres  rating  similarity
                  The Dark Knight Rises [Action, Crime, Drama, Thriller]     7.6       0.412
                          Batman Begins           [Action, Crime, Drama]     7.5       0.353
                         Batman Returns                [Action, Fantasy]     6.6       0.304
                         Batman Forever         [Action, Crime, Fantasy]     5.2       0.292
Batman: The Dark Knight Returns, Part 2              [Action, Animation]     7.9       0.268
                                 Batman                [Fantasy, Action]     7.0       0.225
                         Batman & Robin         [Action, Crime, Fantasy]     4.2       0.214
               Amidst the Devil's Wings           [Drama, Action, Crime]     0.0       0.192


## Step 3: Collaborative Filtering (SVD / Matrix Factorization)

In [6]:
from scipy.sparse.linalg import svds
from scipy.sparse import csr_matrix

np.random.seed(42)

# Simulate user-movie ratings (in real system, this comes from user logs)
n_users  = 500
n_movies = len(df)
sparsity = 0.03
n_ratings = int(n_users * n_movies * sparsity)

user_ids  = np.random.randint(0, n_users, n_ratings)
movie_ids = np.random.randint(0, n_movies, n_ratings)
ratings   = np.random.choice([1,2,3,4,5], n_ratings, p=[0.05,0.10,0.20,0.35,0.30])

R_sparse = csr_matrix((ratings, (user_ids, movie_ids)), shape=(n_users, n_movies))
R_dense  = R_sparse.toarray().astype(float)

# Normalize by subtracting user mean
user_mean = np.true_divide(R_dense.sum(1), (R_dense!=0).sum(1)+1e-9)
R_demeaned = R_dense - user_mean.reshape(-1,1)

# SVD decomposition (k=50 latent factors)
U, sigma, Vt = svds(csr_matrix(R_demeaned), k=50)
sigma = np.diag(sigma)

# Reconstruct predicted ratings
R_predicted = np.dot(np.dot(U, sigma), Vt) + user_mean.reshape(-1,1)
print(f"SVD Matrix Factorization: U{U.shape} × Σ{sigma.shape} × Vt{Vt.shape}")
print("Collaborative Filtering engine ready")

SVD Matrix Factorization: U(500, 50) × Σ(50, 50) × Vt(50, 4803)
Collaborative Filtering engine ready


In [7]:
def collab_recommend(user_id: int, top_n: int = 10):
    """Collaborative: recommend based on what similar users liked"""
    user_pred = R_predicted[user_id]
    already_rated = np.where(R_dense[user_id] > 0)[0]
    candidate_idx = np.argsort(user_pred)[::-1]
    new_recs = [i for i in candidate_idx if i not in already_rated][:top_n]
    return df[["title_x","genres_list","vote_average"]].iloc[new_recs].rename(
        columns={"title_x":"title","genres_list":"genres","vote_average":"rating"}
    ).assign(predicted_rating=user_pred[new_recs].round(2))

print("Collaborative Filtering for User #42:")
print(collab_recommend(42, top_n=8).to_string(index=False))

Collaborative Filtering for User #42:
                           title                                        genres  rating  predicted_rating
                      Auto Focus                                [Drama, Crime]     6.1              0.88
               Jupiter Ascending [Science Fiction, Fantasy, Action, Adventure]     5.2              0.80
                           Birth                    [Drama, Mystery, Thriller]     5.9              0.76
                     Tumbleweeds                               [Comedy, Drama]     6.2              0.75
Superman IV: The Quest for Peace          [Action, Adventure, Science Fiction]     4.1              0.75
Sinbad: Legend of the Seven Seas                [Family, Animation, Adventure]     6.6              0.74
                 The Lone Ranger                  [Action, Adventure, Western]     5.9              0.73
               Playing for Keeps                             [Comedy, Romance]     5.4              0.73


# LIMITATION OF CONTENT BASED: Genre bubble — only recommends similar movies. You will always get action movies if you watch action movies. No discovery. 

# LIMITATION OF COLLABORATIVE: Cold-start problem — new users with no ratings get no personalized recommendations.

## Step 4: Hybrid System (Content + Collaborative)

In [8]:
def hybrid_recommend(user_id: int, title: str, top_n: int = 10,
                     alpha: float = 0.6):
    """
    Hybrid = alpha * content_score + (1-alpha) * collab_score
    alpha=0.6 means 60% content, 40% collaborative
    """
    # Content scores (similarity to given movie)
    if title not in indices:
        return content_recommend(title, top_n)
    idx = indices[title]
    content_scores = cos_sim[idx]

    # Collaborative scores (user preference)
    collab_scores = R_predicted[user_id]
    # Normalize both to [0,1]
    def norm(arr):
        mn, mx = arr.min(), arr.max()
        return (arr - mn) / (mx - mn + 1e-9)
    c_norm  = norm(content_scores)
    co_norm = norm(collab_scores)

    hybrid_scores = alpha * c_norm + (1 - alpha) * co_norm

    already_rated = np.where(R_dense[user_id] > 0)[0]
    candidate_idx = np.argsort(hybrid_scores)[::-1]
    new_recs = [i for i in candidate_idx if i != idx and i not in already_rated][:top_n]

    result = df[["title_x","genres_list","vote_average"]].iloc[new_recs].copy()
    result["hybrid_score"] = hybrid_scores[new_recs].round(3)
    result["content_sim"]  = content_scores[new_recs].round(3)
    result["collab_score"] = collab_scores[new_recs].round(2)
    return result.rename(columns={"title_x":"title","genres_list":"genres","vote_average":"rating"})

print("HYBRID Recommendations (User #42, seed=The Dark Knight):")
print(hybrid_recommend(42, "The Dark Knight", top_n=8).to_string(index=False))

HYBRID Recommendations (User #42, seed=The Dark Knight):
                                  title                               genres  rating  hybrid_score  content_sim  collab_score
Batman: The Dark Knight Returns, Part 2                  [Action, Animation]     7.9         0.415        0.268          0.52
                     The Usual Suspects             [Drama, Crime, Thriller]     8.1         0.384        0.145          0.73
                          Batman Begins               [Action, Crime, Drama]     7.5         0.361        0.353          0.00
               Amidst the Devil's Wings               [Drama, Action, Crime]     0.0         0.347        0.192          0.41
                             Auto Focus                       [Drama, Crime]     6.1         0.346        0.029          0.88
       Superman IV: The Quest for Peace [Action, Adventure, Science Fiction]     4.1         0.341        0.066          0.75
                         Batman Returns                    [A

## Step 5: Evaluation & Comparison

In [10]:
# Evaluation: compare content-based vs hybrid using simulated held-out ratings
np.random.seed(0)
n_eval = 50  # test 50 users
hits_content, hits_hybrid = 0, 0

for user_id in range(n_eval):
    liked = np.where(R_dense[user_id] >= 4)[0]
    if len(liked) == 0: continue
    test_movie_idx = liked[0]
    test_movie = df["title_x"].iloc[test_movie_idx]
    if test_movie not in indices: continue

    # Content-based recs
    cb_recs = content_recommend(test_movie, top_n=10)
    cb_titles = set(cb_recs["title"].tolist())

    # Hybrid recs
    hy_recs = hybrid_recommend(user_id, test_movie, top_n=10)
    hy_titles = set(hy_recs["title"].tolist())

    # Check if other liked movies appear in recommendations
    other_liked = set(df["title_x"].iloc[liked[1:]].tolist())
    hits_content += len(cb_titles & other_liked) > 0
    hits_hybrid  += len(hy_titles & other_liked) > 0

print(f"Hit Rate @ 10 (Content-Based): {hits_content/n_eval:.1%}")
print(f"Hit Rate @ 10 (Hybrid):         {hits_hybrid/n_eval:.1%}")
print(f"Hybrid improved hit rate by +{(hits_hybrid-hits_content)/n_eval:.1%}")
print("""
WHY HYBRID IS BETTER:
  Content-based alone: recommends movies similar to one film
                       → gets stuck in same genre/director
  Collaborative alone: cold-start problem (new users have no history)
  Hybrid:             content = what's similar to your taste
                      + collaborative = what users like you enjoyed
                      Best of both worlds!
""")

Hit Rate @ 10 (Content-Based): 14.0%
Hit Rate @ 10 (Hybrid):         2.0%
Hybrid improved hit rate by +-12.0%

WHY HYBRID IS BETTER:
  Content-based alone: recommends movies similar to one film
                       → gets stuck in same genre/director
  Collaborative alone: cold-start problem (new users have no history)
  Hybrid:             content = what's similar to your taste
                      + collaborative = what users like you enjoyed
                      Best of both worlds!



# WHY HYBRID IS BETTER:
#  Content-based alone: recommends movies similar to one film
#                       → gets stuck in same genre/director
#  Collaborative alone: cold-start problem (new users have no history)
#  Hybrid: content = what's similar to your taste + collaborative = what users like you enjoyed. Best of both worlds!